# 🔺 Delta Lake — Schema Enforcement & Evolution

---

## 📌 Keypoints até agora

Antes de avançar, vamos consolidar o que aprendemos sobre o funcionamento interno do Delta Lake:

---

### 1 — Operações básicas não modificam nem removem arquivos Parquet

```
INSERT  →  novo .parquet criado
UPDATE  →  novo .parquet criado  +  antigo marcado como 'remove' no _delta_log
DELETE  →  .parquet original mantido  +  Deletion Vector (.bin) criado
MERGE   →  combinação das estratégias acima
```

O arquivo físico só é de fato removido quando `VACUUM` ou `OPTIMIZE` são executados.

---

### 2 — O Delta Log é a Fonte Única da Verdade (Single Source of Truth)

O Spark **nunca** lê os `.parquet` diretamente. O fluxo é sempre:

```
  Query SQL
      │
      ▼
 _delta_log  ──► quais arquivos são válidos? (add / remove)
      │       ──► quais linhas ignorar?       (deletionVector)
      │       ──► quais arquivos pular?        (min/max stats)
      │
      ▼
 .parquet(s) selecionados
      │
      ▼
  Resultado
```

---

### 3 — Deletion Vectors controlam deleções no nível de linha

```
Parquet:  [ linha 0: Bob | linha 1: Steve | linha 2: Ray ]
                ↑
         deletion_vector.bin aponta para linha 0 → ignorada na leitura

Resultado da query: [ Steve | Ray ]
```

---

### 4 — O Delta Log mantém estatísticas das primeiras 32 colunas

Cada commit `add` no `_delta_log` armazena `minValues`, `maxValues` e `nullCount` por coluna.  
Isso habilita o **File Skipping**: o Spark descarta arquivos inteiros quando as estatísticas garantem que o filtro não encontrará resultados ali.

```
  WHERE id = 500
      │
      ▼
  arquivo_1.parquet  →  stats: id entre 1   e 100   →  ✗ SKIP
  arquivo_2.parquet  →  stats: id entre 101 e 600   →  ✓ LER
  arquivo_3.parquet  →  stats: id entre 601 e 900   →  ✗ SKIP
```

> ⚠️ Se o schema tiver **colunas aninhadas** (structs), cada subcampo conta como uma coluna separada para o limite de 32 — reduzindo o número de colunas com estatísticas.

---

### 5 — Unity Catalog pode restringir a leitura do Delta Log

Para **tabelas gerenciadas** (managed tables) no Unity Catalog, o acesso direto ao `_delta_log` via `%fs ls` ou `SELECT * FROM json.\`...\`` pode ser bloqueado por políticas de segurança.  
Isso é esperado — use `DESC HISTORY` e `DESC EXTENDED` como alternativas.

---

## ⚡ Quick Review — ACID

O Delta Lake é um formato **ACID-compliant**. Antes de entrar em Schema Enforcement, vale revisitar o que cada propriedade significa na prática:

---
![Foto](./Fotos/1.png)
![Foto](./Fotos/2.png)
![Foto](./Fotos/3.png)
![Foto](./Fotos/4.png)
![Foto](./Fotos/5.png)
![Foto](./Fotos/6.png)

## 🧱 Schema Enforcement & Evolution

O Delta Lake aplica **dois comportamentos complementares** em relação ao schema:

| Comportamento | O que faz | Quando ocorre |
|---|---|---|
| **Schema Enforcement** | Rejeita writes que não respeitam o schema atual | Sempre, por padrão |
| **Schema Evolution** | Adapta o schema para aceitar novos dados | Apenas quando explicitamente habilitado |

```
  Tabela demo:  id INT | valid BOOLEAN | name STRING
                  │
   ┌──────────────┴─────────────────┐
   │                                │
   ▼                                ▼
Schema Enforcement           Schema Evolution
"Você só pode inserir         "Posso expandir o schema
 dados que caibam             para aceitar colunas
 neste schema"                ou tipos novos"
```

---

## 1. Preparação

In [ ]:
%%sql
DROP TABLE IF EXISTS demo;

CREATE OR REPLACE TABLE demo (
  id    INT,
  valid BOOLEAN,
  name  STRING
);

INSERT INTO demo VALUES (1,True,'OK');

---
## 2. Schema Enforcement — verificando na prática

O Schema Enforcement garante que nenhum dado que viole a estrutura da tabela seja aceito.  
Vamos testar os três tipos de violação mais comuns:

### 2.1 — Coluna extra

A tabela tem 3 colunas. Tentamos inserir 4 valores:

In [ ]:
%%sql
-- ❌ Erro: número de colunas não corresponde ao schema
INSERT INTO demo VALUES (1,True,'OK','Or not')

> **Erro esperado:** `AnalysisException` — o número de colunas do VALUES não corresponde ao schema da tabela.  
> O Delta Lake rejeita a operação **antes mesmo de gravar qualquer arquivo**.

### 2.2 — Tipo incompatível (sem cast possível)

A coluna `id` é `INT`. Tentamos inserir uma `String`:

In [ ]:
%%sql
-- ❌ Erro: tipo incompatível — 'one' não pode ser convertido para INT
INSERT INTO demo VALUES ('one',True,'OK')

> **Erro esperado:** erro de tipo e cast — o Spark tenta converter `'one'` para `INT`, falha, e bloqueia a operação.

### 2.3 — Insert válido

In [ ]:
%%sql
-- ✅ Tipos corretos, número de colunas correto
INSERT INTO demo VALUES (1,True,'OK')

### 2.4 — Overflow de tipo

O tipo `INT` suporta valores até `2.147.483.647`. Um número maior é inferido como `BIGINT`:

In [ ]:
%%sql
-- ❌ Erro: literal 100000000000 é BIGINT, incompatível com a coluna INT
INSERT INTO demo VALUES (100000000000,True,'OK')

> **Erro esperado:** `BIGINT` não pode ser implicitamente reduzido para `INT` pois haveria perda de dados.  
> Isso demonstra que o enforcement opera no nível do **tipo de dado**, não apenas da estrutura.

---

### Resumo — Schema Enforcement

```
  Tipo de violação          Resultado
  ─────────────────────     ────────────────────────────────
  Coluna a mais             AnalysisException (estrutura)
  Tipo incompatível         Cast error (impossível converter)
  Overflow de tipo          Type mismatch (BIGINT → INT)
  Tudo correto              ✅ Insert aceito
```

---
## 3. Schema Evolution — habilitando por operação

Quando um write **precisa** adicionar colunas novas ao schema existente, o Delta Lake permite isso via `mergeSchema`.  
A operação só expande o schema se os tipos das colunas existentes forem compatíveis.

### 3.1 — Criando o DataFrame com coluna extra

In [ ]:
%python
data= [(1,True,'OK','Or not')]
df = spark.createDataFrame(data, schema='id Int,valid Boolean,name String,suprise String')
df.display

> O DataFrame tem 4 colunas (`id`, `valid`, `name`, `suprise`).  
> Um `INSERT INTO demo` simples falharia — mas com `mergeSchema` o Delta Lake expande o schema da tabela para acomodar a nova coluna.

### 3.2 — Gravando com `mergeSchema`

In [ ]:
%python
df.write\
.option('mergeSchema','true')\
.mode('append')\
.saveAsTable('demo')

In [ ]:
%%sql
-- A coluna 'suprise' agora faz parte do schema
-- Linhas antigas têm NULL nessa coluna
SELECT * FROM demo

> **O que aconteceu internamente:**  
> O Delta Lake comparou o schema do DataFrame com o schema da tabela, identificou `suprise` como coluna nova,  
> adicionou-a ao schema da tabela no `_delta_log` (campo `metaData`) e gravou a linha.  
> As linhas anteriores recebem `NULL` na coluna nova automaticamente.

```
  Schema antes:  id INT | valid BOOLEAN | name STRING
  Schema depois: id INT | valid BOOLEAN | name STRING | suprise STRING
                                                           ↑
                                                     adicionada pelo mergeSchema
```

---
## 4. Limites do Schema Evolution

O `mergeSchema` **não** permite mudar o tipo de uma coluna existente.  
Tentar inserir `id` como `String` em uma coluna `INT` falha mesmo com a opção habilitada:

In [ ]:
%python
data= [('one',True,'OK','Or not')]
df = spark.createDataFrame(data, schema='id String,valid Boolean,name String,suprise String')
df.write\
.option('mergeSchema','true')\
.mode('append')\
.saveAsTable('demo')

> **Erro esperado:** o `mergeSchema` detecta que a coluna `id` já existe com tipo `INT` e o DataFrame traz `STRING`.  
> Mudança de tipo é **incompatível** — a operação é abortada e a tabela permanece inalterada.

```
  O que mergeSchema PODE fazer          O que mergeSchema NÃO PODE fazer
  ──────────────────────────────         ────────────────────────────────
  Adicionar colunas novas        ✅       Mudar tipo de coluna existente  ❌
  Adicionar campos em structs    ✅       Renomear coluna existente       ❌
  Aceitar NULL em colunas novas  ✅       Remover coluna existente        ❌
```

---
## 5. Schema Evolution para toda a sessão Spark

Em vez de passar `mergeSchema` a cada operação, é possível habilitar a evolução automática para **toda a sessão Spark**.  
A partir daí, qualquer write — inclusive via SQL — aceita colunas novas sem precisar da opção explícita.

### 5.1 — Habilitando autoMerge

In [ ]:
%%sql
SET spark.databricks.delta.schema.autoMerge.enabled = True

Com `autoMerge` ativo, um `INSERT INTO` SQL com mais colunas do que o schema atual **não falha** — o schema é expandido automaticamente:

In [ ]:
%%sql
INSERT INTO demo VALUES (1,True, 'OK','Extra','Super');

In [ ]:
%%sql
-- O schema foi expandido para incluir as novas colunas automaticamente
SELECT * FROM demo;

> **Atenção:** `autoMerge` é conveniente, mas pode ser perigoso em pipelines de produção —  
> um dado malformado pode silenciosamente expandir o schema da tabela sem que ninguém perceba.  
> Prefira `mergeSchema` por operação quando possível.

```
  autoMerge = False  (padrão)
  → Schema Enforcement: qualquer coluna extra → ❌ erro

  autoMerge = True
  → Schema Evolution automática: coluna extra → ✅ schema expandido
  → Mudança de tipo ainda → ❌ erro
```

---
## 6. Sobrescrevendo o Schema — `overwriteSchema`

Quando a necessidade é **substituir completamente** o schema da tabela, use `overwriteSchema`.  
Diferente do `mergeSchema` (que expande), o `overwriteSchema` **descarta** o schema antigo e adota o novo.

> ⚠️ Isso destrói o schema atual e todos os dados da tabela — use com cuidado.

In [ ]:
%python
data = [('One', 'The','New')]
df = spark.createDataFrame(data,schema='id String, c1 String, c2 String')
df.write\
.option('overwriteSchema','true')\
.mode('overwrite')\
.saveAsTable('demo')

In [ ]:
%%sql
-- Schema completamente substituído: id STRING | c1 STRING | c2 STRING
-- Dados anteriores foram removidos
SELECT * FROM demo;

> **O que aconteceu:**  
> O `overwriteSchema` gravou um novo `metaData` no `_delta_log` com o schema completamente novo  
> e os dados anteriores foram descartados (mode `overwrite`).

---

### Resumo — Comparação das estratégias

```
┌───────────────────────┬──────────────────────────────────────────────────┐
│ Opção                 │ Comportamento                                    │
├───────────────────────┼──────────────────────────────────────────────────┤
│ Padrão (sem opção)    │ Schema Enforcement — rejeita qualquer divergência│
│ mergeSchema = true    │ Adiciona colunas novas; mantém dados anteriores  │
│ autoMerge = true      │ Igual ao mergeSchema, mas para toda a sessão     │
│ overwriteSchema = true│ Substitui schema e dados completamente           │
└───────────────────────┴──────────────────────────────────────────────────┘

  Em todos os casos: mudança de tipo em coluna existente → ❌ sempre bloqueado
```